# Hybrid RAG

This notebook continues from `02_7_vectordb_docker.ipynb`. In 02_7 it was mentioned that the RAG approach we used (vector search) was simple and that we would use a more complicated one here. For example, in 02_7 we queried against the entire database which is time-consuming and memory intensive. 

Instead if we e.g. knew that we are only interested in albums with a (review) score of 8.0 or above, we could filter out lower-scoring albums to save time and tokens. Or we could only query against certain music genres or certain years. 

Thus we will be using **hybrid retrieval** by progressively improving the RAG pipeline across three techniques:

1. **Keyword filter** — restrict the candidate pool to documents that contain a specific term
2. **Metadata filter** — use structured fields (score, year, artist) stored natively in ChromaDB
3. **LLM re-ranking** — retrieve a larger set, then let the model promote the most relevant

We run the **same sample query** at each stage so you can compare results directly.

**Prerequisites:** Complete `02_7` first — ChromaDB must be running with the `pitchfork_reviews` collection loaded.

## Setup

In [ ]:
%load_ext dotenv
%dotenv ../../05_src/.env
%dotenv ../../05_src/.secrets
import sys
sys.path.append('../../05_src/')

In [10]:
from utils.clients import get_client
from IPython.display import display, Markdown
import chromadb
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction
import json
import os

client = get_client(use_gateway=True)
MODEL = os.getenv('MODEL')
EMBEDDING_MODEL = os.getenv('EMBEDDING_MODEL')
COLLECTION_NAME = 'pitchfork_reviews'
CHROMA_URL = os.getenv('CHROMA_URL')

### Helper Functions

These functions are brought forward from `02_7`. `get_context_data()` queries ChromaDB and returns enriched result dicts (text + metadata). `generate_prompt()` builds the prompt with album context. `generate_response()` calls the model.

Notice the keywords `where` and `where_document`. Their uses are discussed [here](https://docs.trychroma.com/docs/querying-collections/full-text-search).

In [11]:
def get_context_data(query: str, collection: chromadb.api.models.Collection, top_n: int,
                     where_document: dict = None, where: dict = None):
    kwargs = dict(query_texts=[query], n_results=top_n)
    if where_document:
        kwargs['where_document'] = where_document
    if where:
        kwargs['where'] = where
    results = collection.query(**kwargs)
    context_data = []
    for idx in range(len(results['ids'][0])):
        details = dict(results['metadatas'][0][idx])
        details['text'] = results['documents'][0][idx]
        context_data.append(details)
    return context_data

def generate_prompt(query: str, context_data: list):
    top_n = len(context_data)
    prompt = f'Given a query, provide a detailed response using the context from relevant Pitchfork reviews. The context will contain references to {top_n} album reviews.\n\n'
    prompt += 'The score is numeric and its scale is from 0 to 10, with 10 being the highest rating. Any album with a score greater than 8.0 is considered a must-listen; album with a score greater than 6.5 is good.\n\n'
    prompt += f'<query>{query}</query>\n\n'
    prompt += '<context>\n'
    for k, context in enumerate(context_data):
        prompt += f'<album {k}>\n'
        prompt += f"- Album Title: {context.get('album', 'N/A')}\n"
        prompt += f"- Album Artist: {context.get('artist', 'N/A')}\n"
        prompt += f"- Album Score: {context.get('score', 'N/A')}\n"
        prompt += f"- Genre: {context.get('genre', 'N/A')}\n"
        prompt += f"- Label: {context.get('label', 'N/A')}\n"
        prompt += f"- Release Year: {context.get('year', 'N/A')}\n"
        prompt += f"- Review Quote: {context.get('text', 'N/A')}\n"
        prompt += f'</album {k}>\n\n'
    prompt += '</context>\n\n'
    prompt += '\nBased on the context and nothing else, provide a detailed response to the query.'
    return prompt

def generate_response(query: str, collection: chromadb.api.models.Collection, top_n: int = 3,
                      where_document: dict = None, where: dict = None):
    context_data = get_context_data(query, collection, top_n,
                                    where_document=where_document, where=where)
    prompt = generate_prompt(query, context_data)
    response = client.responses.create(
        model=MODEL,
        instructions='You are a helpful assistant that provides information based on Pitchfork reviews.',
        input=[{'role': 'user', 'content': prompt}],
        max_output_tokens=500,
        temperature=1.0
    )
    return response.output_text

### Connect to ChromaDB

Set `USE_GATEWAY = True` if you are using the API gateway instead of a direct OpenAI key.

In [12]:
USE_GATEWAY = os.getenv('USE_GATEWAY', 'False').lower() == 'true'

chroma = chromadb.HttpClient(host=CHROMA_URL)
if USE_GATEWAY:
    embedding_function = OpenAIEmbeddingFunction(
        api_key='any value',
        api_base='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
        api_type='openai',
        model_name=EMBEDDING_MODEL,
        default_headers={'x-api-key': os.getenv('API_GATEWAY_KEY')}
    )
else:
    embedding_function = OpenAIEmbeddingFunction(
        api_key=os.getenv('OPENAI_API_KEY'),
        model_name=EMBEDDING_MODEL
    )

collection = chroma.get_collection(
    name=COLLECTION_NAME,
    embedding_function=embedding_function
)
print(f'Collection loaded: {collection.count():,} documents')

Collection loaded: 18,672 documents


## Section 1: Baseline — Pure Vector Search

Before we improve retrieval, let's establish a baseline using the pure vector pipeline from `02_7`. We embed the query, find the 3 most similar review chunks by cosine similarity, and generate a response.

We will reuse the **same query** throughout every section so you can directly compare results.

In [13]:
SAMPLE_QUERY = 'What are some highly rated albums by emerging dream pop artists?'

In [14]:
baseline_response = generate_response(SAMPLE_QUERY, collection, top_n=3)
display(Markdown(baseline_response))

If you’re looking for highly rated albums by emerging dream pop artists, there are some standout releases that have garnered acclaim in the dream pop and related genres. Here are three albums worth checking out:

1. **"The Dream of a Modern Day" by Mahogany** - This album received an impressive score of **8.4**, making it a must-listen for fans of the genre. The album opens with a droning string quartet that seamlessly transitions into vibrant pop elements, showcasing intricate production and lush instrumentation. Mahogany's sound is characterized by ethereal vocals and a rich palette of sounds, weaving together everything from complex percussion to sweeping orchestration. The track "Chance" is particularly notable for its exuberance, while "Red Marrow, His Sorrow" reveals a more introspective side with its minimalist arrangement. This debut full-length promises significant potential for Mahogany as they continue to carve a niche in the dream pop landscape.

2. **"or you could just go through your whole life and be happy anyway" by Aarktica** - Scoring **8.1**, this album is part of the Darla label's Bliss Out series and is described as an essential release that encapsulates the dreamy and ambient qualities of dream pop. Aarktica's Jon De Rosa merges strings and electronic beats to create tranquil atmospheres, channeling feelings of contemplative melancholy. Tracks like "Aura Lee" build from layered instrumentation and vocal harmonies, inviting listeners into a serene sonic world. The delicate production touches reflect not just a mastery of sound but also an emotional depth that resonates throughout the seven tracks.

3. **"Low Level Owl Vol. I & II" by The Appleseed Cast** - Though this album straddles the boundary between rock and dream pop, it has received a remarkable score of **9.0**, making it another significant listen. The extensive double album features intricate arrangements and thematic coherence that connect its various instrumental and vocal pieces. Drawing inspiration from a range of influences, The Appleseed Cast masterfully blends lush atmospherics with engaging melodies. Their work offers expansive soundscapes reminiscent of both dream pop and post-rock, featuring lengthy tracks that evoke a sense of journey and introspection.

Each of these albums showcases the evolving nature of dream pop and related genres, presenting sounds that are lush, atmospheric, and emotionally resonant. Whether you're exploring the ornate melodies of Mahogany, the tranquil tones of Aarktica, or the expansive sound

**What the baseline does:**
- Converts the query to a vector using the same embedding model as the corpus
- Returns the 3 chunks most similar by cosine distance

**What it doesn't do:**
- It has no knowledge of exact words in the query — `dream pop` is treated semantically, not literally (so results may be returned because they contain words with similar meaning to 'dream pop' even though they aren't actually dream pop; or they may not be related to dream pop at all, even semantically, if other text in the review is responsible for the similar embedding btwn query and result -- a problem of emphasis)
- It ignores structured fields like critic score or release year
- It returns the closest chunks by embedding distance, which may not be the *most useful* for the query

The next three sections address each of these gaps.

## Section 2: Keyword Filter (`where_document`)

ChromaDB's `where_document={"$contains": keyword}` filter restricts the search to documents whose **text contains the keyword** before similarity ranking is applied.

**When this is useful:**
- The query involves **exact terms** that must appear: artist names, album titles, subgenre labels
- You want **lexical precision** — you need the word to literally be there

**When it hurts:**
- The keyword is too broad or semantically close to the query → no visible difference in results
- The keyword is very rare → too few candidates remain

> **Key insight:** Even when a keyword matches the query topic (like `'dream pop'` here), the filter still adds value by requiring the phrase to *literally appear* in each chunk. Albums that sound like dream pop but are reviewed as 'shoegaze' or 'indie rock' will be excluded — the vector search finds them semantically, but the keyword filter would not. An artist name like `'Mazzy Star'` or a label name like `'4AD'` would steer results in a completely different direction.

In [15]:
def keyword_search(query: str, collection: chromadb.api.models.Collection,
                   keyword: str, top_n: int = 3):
    """Vector search restricted to documents containing `keyword`."""
    return get_context_data(
        query, collection, top_n,
        where_document={'$contains': keyword}
    )

In [16]:
keyword_results = keyword_search(SAMPLE_QUERY, collection, keyword='dream pop', top_n=3)

for i, r in enumerate(keyword_results):
    print(f"[{i}] {r.get('artist')} — {r.get('album')} ({r.get('year')}) Score: {r.get('score')}")

[0] psapp — tiger, my friend (2004) Score: 7.5
[1] landing — oceanless (2001) Score: 8.0
[2] say lou lou — lucid dreaming (2015) Score: 5.8


Note these responses actually have lower grades than the results from straight vector search; that's because here you are restricting only to results containing the literal phrase 'dream pop'. 

This is a tradeoff and can sometimes even be a wrong decision - e.g. if you are filtering for literal string matches there may be synonyms, alternative spellings, misspellings that would be erronously excluded.

In [17]:
keyword_response = generate_response(SAMPLE_QUERY, collection, top_n=3,
                                     where_document={'$contains': 'dream pop'})
display(Markdown(keyword_response))

If you're looking for highly rated albums by emerging dream pop artists, a couple of notable recommendations stand out:

1. **Oceanless by Landing (Score: 8.0)**
   - **Released:** 2001
   - Landing's *Oceanless* is firmly rooted in the dream pop genre with a distinctive fuzzed-out drone-pop style. The Pitchfork review emphasizes the album's ability to blend the ethereal with the earthy, showcasing how carefully manipulated feedback can create beautiful soundscapes. The lengthy compositions, some exceeding 20 minutes, allow the listener to dive deep into a textured experience. Landing's sound is described as reminiscent of nature, as they channel ambient forces, making it a compelling listen for anyone exploring the dream pop realm.

2. **Tiger, My Friend by Psapp (Score: 7.5)**
   - **Released:** 2004
   - Although slightly below the must-listen threshold, Psapp's debut *Tiger, My Friend* offers a unique take on dream pop. The review notes their playful instrumentation, with elements like glockenspiels and childlike melodies, creating an airy, whimsical atmosphere. Psapp's use of intricate rhythms combined with breathy vocals provides a refreshing contrast to more emotionally heavy dream pop, appealing to those who appreciate subtlety in music.

While *Lucid Dreaming* by Say Lou Lou (Score: 5.8) is also mentioned, it didn't receive a high enough score to be recommended based on Pitchfork's scale. The review critiques the album for being formulaic and heavily reliant on '80s revival sounds, indicating it might not satisfy those looking for a standout dream pop experience.

In summary, for a captivating dive into the dream pop genre from emerging artists, *Oceanless* by Landing and *Tiger, My Friend* by Psapp should be at the top of your listening list.

**Compare with baseline:** The baseline used semantic similarity, so it may include dream pop-adjacent albums that don't use the phrase *dream pop* in their review text. With `keyword='dream pop'`, every result chunk must contain that phrase — albums described as 'shoegaze' or 'indie folk' without the dream pop label are excluded even if they sound similar.

This illustrates the key trade-off: keyword filtering gives **lexical precision** but sacrifices recall for albums described using synonymous terms.

**Try other keywords:**
- An artist name (`'Mazzy Star'`, `'Beach House'`) to see only their reviews
- A label name (`'4AD'`, `'Sub Pop'`) to filter by imprint
- A technical term (`'reverb'`, `'ethereal'`) to steer toward a different sonic texture
- `'shoegaze'` — notice this produces *different* results from the baseline and from `'dream pop'`, because it finds review chunks that use the word shoegaze even if the album is not specifically dream pop

## Section 3: Metadata Filtering (`where`)

When we loaded the collection in `02_7`, we stored structured fields alongside each chunk:

| Field | Type | Example |
|-------|------|---------|
| `score` | float | `8.7` |
| `year` | int | `2001` |
| `artist` | str | `'Radiohead'` |
| `album` | str | `'Kid A'` |
| `genre` | str | `'Electronic, Rock'` |
| `label` | str | `'Capitol'` |

ChromaDB's `where` parameter filters on these fields **before** ranking. Supported operators: `$eq`, `$ne`, `$gt`, `$gte`, `$lt`, `$lte`, `$and`, `$or`, `$in`.

In [18]:
def metadata_search(query: str, collection: chromadb.api.models.Collection,
                    where_filter: dict, top_n: int = 3):
    """Vector search restricted to documents matching `where_filter` on metadata fields."""
    return get_context_data(query, collection, top_n, where=where_filter)

**Example 1: Score filter** — only reviews rated above 8.0

In [19]:
score_results = metadata_search(
    SAMPLE_QUERY, collection,
    where_filter={'score': {'$gt': 8.0}},
    top_n=3
)

for i, r in enumerate(score_results):
    print(f"[{i}] {r.get('artist')} — {r.get('album')} ({r.get('year')}) Score: {r.get('score')}")

[0] aarktica — or you could just go through your whole life and be happy anyway (2002) Score: 8.1
[1] mahogany — the dream of a modern day (0) Score: 8.4
[2] the appleseed cast — low level owl vol. i & ii (2001) Score: 9.0


In [20]:
score_response = generate_response(SAMPLE_QUERY, collection, top_n=3,
                                   where={'score': {'$gt': 8.0}})
display(Markdown(score_response))

If you're looking for highly rated albums by emerging dream pop artists, there are a couple of standout choices from the context of Pitchfork reviews that you should consider:

1. **"The Dream of a Modern Day" by Mahogany (Score: 8.4)**  
   This debut album from Mahogany showcases an extraordinary blend of dreamy pop and intricate orchestration. The album begins with "Movement I," a captivating string quartet interlude that flows into the vibrant track "Chance." The production shines throughout, featuring an impressive array of instruments that create a lush soundscape, combining ride cymbals, strings, and jangly guitars. The ethereal vocals of Massais are a highlight, resonating beautifully with the complex musical arrangements. The album includes standout moments like "Soleil Radieux," where intricate percussion meets rich orchestration, and "Vista-dome," combining powerful rhythms with haunting melodies. This release not only marks a significant entry for the band but also points to a promising future in the dream pop genre.

2. **"or you could just go through your whole life and be happy anyway" by Aarktica (Score: 8.1)**  
   Aarktica's album embodies a meditative and atmospheric dream pop sound. With an exploration of melancholy themes through sonic layers of strings and electronic beats, Jon De Rosa crafts a captivating auditory experience. The album is particularly noted for its rich textures and the way it evokes imagery of wintry landscapes. Tracks like "Aura Lee" build gradually into a lush, swirling crescendo, showcasing De Rosa’s compositional expertise. The dreamlike quality of the music, paired with his soft, breathy vocals, effectively immerses the listener in a world of introspection.

3. **"Low Level Owl Vol. I & II" by The Appleseed Cast (Score: 9.0)**  
   While more aligned with post-rock, The Appleseed Cast's ambitious double album incorporates elements often found in dream pop, characterized by lush production and atmospheric soundscapes. With a duration of over an hour and forty-five minutes, the album gracefully blends various influences while maintaining a cohesive identity. The review highlights the intricate arrangements that evoke the sounds of bands like My Bloody Valentine and the Flaming Lips, making this project essential listening for those seeking immersive experiences in music. Though not strictly classified as dream pop, its expansive sound makes it appealing for fans of the genre.

These albums represent some of the best in

**Example 2: Compound filter** — high-scoring albums released from 2003 onwards

In [ ]:
compound_results = metadata_search(
    SAMPLE_QUERY, collection,
    where_filter={'$and': [{'score': {'$gt': 8.0}}, {'year': {'$gte': 2003}}]}, #could do or instead of and, depending on what you want 
    top_n=3
)

for i, r in enumerate(compound_results):
    print(f"[{i}] {r.get('artist')} — {r.get('album')} ({r.get('year')}) Score: {r.get('score')}")

[0] parsley sound — parsley sounds (2003) Score: 8.8
[1] various artists — dusk at cubist castle (2004) Score: 9.4
[2] pluramon — dreams top rock (2003) Score: 8.5


In [22]:
compound_response = generate_response(
    SAMPLE_QUERY, collection, top_n=3,
    where={'$and': [{'score': {'$gt': 8.0}}, {'year': {'$gte': 2003}}]}
)
display(Markdown(compound_response))

Here are some highly rated albums by emerging dream pop artists that you should consider:

1. **Parsley Sounds - *parsley sounds***  
   - **Score:** 8.8  
   - **Release Year:** 2003  
   - This debut album from the duo Parsley Sound stands out for its unique blend of melancholic melodies and imaginative production. The album features tracks like "Twilight Mushrooms," where ethereal lyrics are paired with dreamy instrumentals, creating a captivating listening experience. The standout track "Ocean House" offers a meditative psychedelic quality reminiscent of The Beatles, while "Ease Yourself and Glide" exhibits a catchy, lo-fi pop charm inspired by Elliott Smith. The album is noted for its cohesive sound, presenting a fully realized aesthetic that could influence the indie scene significantly.

2. **Various Artists - *dusk at cubist castle***  
   - **Score:** 9.4  
   - **Release Year:** 2004  
   - This collection from the Elephant Six collective, particularly through the work of The Olivia Tremor Control, showcases an impressive integration of classic pop and psychedelic sounds. The album is celebrated for its emotional depth, eschewing mere experimentation for heartfelt songwriting that connects deeply with listeners. With echoes of The Beatles and Beach Boys, *Dusk at Cubist Castle* achieves a remarkable balance between innovative sounds and lasting musical impact. Its ability to evoke human memory and emotion makes it a quintessential listen for fans of the genre.

3. **Pluramon - *dreams top rock***  
   - **Score:** 8.5  
   - **Release Year:** 2003  
   - This album presents a fascinating collaboration between Julee Cruise and producer Markus Schmickler, blending dreamy vocals with layers of electronic textures and shoegaze influences. Tracks like "Time For a Lie" exhibit a compelling contrast between soft melodies and harsh guitar distortions, capturing a unique dream pop atmosphere. The album is characterized by its experimental approach, with moments of melodic beauty interspersed with the raw chaotic sounds described as "breathtaking." Notably, *Dreams Top Rock* is recognized as a fantastic leftfield experiment, perfect for those who appreciate a bold take on the dream pop genre.

These albums not only showcase the emerging talent in dream pop but also highlight innovative approaches and a commitment to emotional resonance, making them essential listens for fans of the genre.

**A note on genre filtering:**

You might expect to filter by genre with `{'genre': {'$eq': 'Rock'}}`. This works, but only for reviews with *exactly* that genre string. In our dataset, genres are stored as comma-separated strings (e.g., `'Electronic, Rock'`), so `$eq 'Rock'` won't match multi-genre entries.

This is a **schema design trade-off**: storing the primary genre as a separate field (`primary_genre`) would enable clean equality filtering. As a workaround, you can use `artist` or `year` filters (which are single-valued) alongside the semantic query.

**Try it:** Change the `score` threshold or `year` cutoff and observe how the candidate set changes. A very strict filter (e.g., `score > 9.5`) may return no results — the function returns an empty list gracefully.

## Section 4: LLM Re-ranking

Vector similarity finds documents that are *semantically close* to the query, but closeness in embedding space doesn't always mean *most useful for answering the question*.

**Re-ranking** separates retrieval from relevance scoring:
1. Retrieve a **larger candidate set** (e.g., top 10) with vector search
2. Ask the language model to **rank candidates by relevance** to the query
3. Return the top-k from the re-ranked list

This adds one API call but can meaningfully improve result quality — especially when the top-1 vector result is semantically close but not the most informative answer.

In [23]:
def llm_rerank(context_data: list, query: str, top_k: int = 3) -> list:
    """Re-rank context_data by relevance to query using a single LLM call."""
    if not context_data:
        return context_data

    # Build a compact candidate list for the model
    candidates_text = ''
    for i, c in enumerate(context_data):
        snippet = c.get('text', '')[:200].replace('\n', ' ')
        candidates_text += (
            f"[{i}] Artist: {c.get('artist', 'N/A')}, "
            f"Album: {c.get('album', 'N/A')}, "
            f"Score: {c.get('score', 'N/A')}, "
            f"Genre: {c.get('genre', 'N/A')}\n"
            f"    Excerpt: {snippet}\n\n"
        )

    rerank_prompt = (
        f'You are ranking album review candidates by how well they answer the user query.\n'
        f'Return ONLY a JSON array of candidate indices ordered from most to least relevant.\n'
        f'Example: [2, 0, 4, 1, 3]\n\n'
        f'Query: {query}\n\n'
        f'Candidates:\n{candidates_text}'
    )

    response = client.responses.create(
        model=MODEL,
        instructions='Return only a valid JSON array of integers. No explanation.',
        input=[{'role': 'user', 'content': rerank_prompt}],
        max_output_tokens=100,
        temperature=0.0
    )

    try:
        raw = response.output_text.strip()
        # Extract the JSON array even if the model wraps it in markdown
        start = raw.find('[')
        end = raw.rfind(']') + 1
        ranked_indices = json.loads(raw[start:end])
        # Filter valid indices and deduplicate
        valid = [i for i in ranked_indices if isinstance(i, int) and 0 <= i < len(context_data)]
        seen = set()
        deduped = [i for i in valid if not (i in seen or seen.add(i))]
        reranked = [context_data[i] for i in deduped]
        # Append any missing candidates at the end
        missing = [c for i, c in enumerate(context_data) if i not in seen]
        return (reranked + missing)[:top_k]
    except Exception:
        # Fallback: return original order
        return context_data[:top_k]

In [24]:
# Retrieve top-10 candidates, then re-rank to top-3
candidates = get_context_data(SAMPLE_QUERY, collection, top_n=10)

print('--- Before re-ranking (top 3 of 10 by vector similarity) ---')
for i, r in enumerate(candidates[:3]):
    print(f"[{i}] {r.get('artist')} — {r.get('album')} ({r.get('year')}) Score: {r.get('score')}")

reranked = llm_rerank(candidates, SAMPLE_QUERY, top_k=3)

print('\n--- After re-ranking ---')
for i, r in enumerate(reranked):
    print(f"[{i}] {r.get('artist')} — {r.get('album')} ({r.get('year')}) Score: {r.get('score')}")

--- Before re-ranking (top 3 of 10 by vector similarity) ---
[0] aarktica — or you could just go through your whole life and be happy anyway (2002) Score: 8.1
[1] mahogany — the dream of a modern day (0) Score: 8.4
[2] the appleseed cast — low level owl vol. i & ii (2001) Score: 9.0

--- After re-ranking ---
[0] mahogany — the dream of a modern day (0) Score: 8.4
[1] parsley sound — parsley sounds (2003) Score: 8.8
[2] pluramon — dreams top rock (2003) Score: 8.5


In [25]:
reranked_response = generate_prompt(SAMPLE_QUERY, reranked)
final_reranked = client.responses.create(
    model=MODEL,
    instructions='You are a helpful assistant that provides information based on Pitchfork reviews.',
    input=[{'role': 'user', 'content': reranked_response}],
    max_output_tokens=500,
    temperature=1.0
).output_text
display(Markdown(final_reranked))

If you're looking for highly rated albums by emerging dream pop artists, there are three standout records that are definitely worth listening to:

1. **Mahogany - *The Dream of a Modern Day* (Score: 8.4)**  
   This debut album showcases Mahogany's remarkable ability to blend dreamy pop and R&B elements. The opening track captures listeners with an unexpected blend of droning strings and lively percussion, leading into the ethereal "Chance." The production is a key highlight, particularly on tracks like "Soleil Radieux" and "Vista-dome," which offer a rich tapestry of sound featuring complex percussion, glowing strings, and jangly guitars. Critics have noted the graceful vocals of Massais, whose delivery is both silky and profound. The album closes with "Synchromie No. 1," an exploration of lush beauty and complex soundscapes, solidifying Mahogany as an exciting new force in dream pop.

2. **Parsley Sound - *Parsley Sounds* (Score: 8.8)**  
   This record is lauded for its immersive, dreamy compositions that blend various musical textures without feeling disjointed. Tracks like "Twilight Mushrooms" and "Ease Yourself and Glide" exemplify the album's successful melding of melancholy and catchy melodies. The participation of elements such as Wurlitzer chords and shimmering guitars create a unique sonic experience reminiscent of past influences while sounding distinctly fresh. This Watford-based duo delivers an impressive debut brimming with inspiration and creativity, likely setting a new standard for indie rock and dream pop alike.

3. **Pluramon - *Dreams Top Rock* (Score: 8.5)**  
   Bringing a unique approach to the dream pop genre, Pluramon's collaboration with Julee Cruise merges ethereal vocals with thick layers of guitar distortion. This album channels the essence of shoegaze while integrating a modern, electronic twist. Listening to “Time For a Lie” reveals the clever editing that keeps the album concise yet impactful. Tracks like "Noise Academy" illustrate a beautiful juxtaposition of serene vocal lines against harsh, experimental noise, evoking a psychedelic ambiance. Critics have praised the album for its adventurous spirit and depth, making it a fantastic unexpected gem in the dream pop landscape.

These albums not only exceed the threshold of 8.0 but also present fresh takes on the dream pop genre, promising to captivate fans of ethereal soundscapes and intricate melodies.

**Cost trade-off:**

Re-ranking adds one extra LLM call — but it is a *short* call: the input is a compact candidate list and the output is just a list of indices. At course scale (10 candidates, ~200 chars each), this is roughly 300–500 tokens, which is negligible.

At production scale with thousands of candidates, you would use a dedicated re-ranking model (e.g., Cohere Rerank, cross-encoders) rather than a generative LLM — but the principle is the same.

## Section 5: Combined Pipeline (`hybrid_rag`)

Now we compose all three techniques into a single function:

1. **Keyword filter** (`where_document`) — restrict to documents mentioning the keyword
2. **Metadata filter** (`where`) — enforce quality or recency constraints
3. **LLM re-ranking** — promote the most relevant from the narrowed candidate set

Each layer trades **recall** for **precision**: the keyword and metadata filters reduce the candidate pool, then re-ranking refines the order. The net effect is higher-quality results for queries where the constraints are meaningful.

In [26]:
def hybrid_rag(
    query: str,
    collection: chromadb.api.models.Collection,
    keyword: str = None,
    where_filter: dict = None,
    top_n_candidates: int = 10,
    top_k_final: int = 3
) -> list:
    """
    Hybrid retrieval pipeline:
      1. ChromaDB vector search with optional keyword + metadata filters
      2. LLM re-ranking to top_k_final
    Returns a list of context_data dicts.
    """
    where_document = {'$contains': keyword} if keyword else None
    candidates = get_context_data(
        query, collection, top_n_candidates,
        where_document=where_document,
        where=where_filter
    )
    if not candidates:
        print('Warning: no candidates returned for the given filters.')
        return []
    return llm_rerank(candidates, query, top_k=top_k_final)

**Query 1 (SAMPLE\_QUERY):** Highly rated dream pop albums — keyword + score filter + re-ranking

In [27]:
results_1 = hybrid_rag(
    query=SAMPLE_QUERY,
    collection=collection,
    keyword='dream pop',
    where_filter={'score': {'$gt': 8.0}},
    top_n_candidates=10,
    top_k_final=3
)

response_1 = generate_prompt(SAMPLE_QUERY, results_1)
answer_1 = client.responses.create(
    model=MODEL,
    instructions='You are a helpful assistant that provides information based on Pitchfork reviews.',
    input=[{'role': 'user', 'content': response_1}],
    max_output_tokens=500,
    temperature=1.0
).output_text
display(Markdown(answer_1))

It seems there are no specific albums or reviews in the provided context to reference. However, I can provide general information about some emerging dream pop artists that have been well-received in the genre, as well as characteristics that make their albums noteworthy.

Dream pop is known for its ethereal soundscapes, lush instrumentation, and often introspective lyrics. Emerging artists in this genre have been gaining attention for their fresh takes on the sound.

1. **Artists to Look Out For**:
   - **Crumb**: This band has been noted for their blend of dreamy melodies and intricate rhythms. Their debut album received positive reviews for its unique sound and captivating arrangements.
   - **Beach House**: Though no longer considered "emerging," their newer albums continue to influence and inspire up-and-coming dream pop artists, often rated highly for their atmospheric quality.
   - **Alvvays**: Known for their catchy hooks wrapped in dreamy instrumentation, their sophomore effort has garnered significant acclaim.

2. **Characteristics of Highly Rated Albums**:
   - **Lush Production**: Successful dream pop albums often feature layered textures and rich sonic landscapes that immerse the listener.
   - **Melodic Hooks**: While ambiance is essential, memorable melodies are also vital for making a lasting impact.
   - **Emotional Weight**: Lyrically, these albums tend to explore themes of nostalgia, love, and introspection, resonating deeply with listeners.

3. **Performance and Innovation**: Emerging artists pushing the boundaries of dream pop include innovative sound design or collaborations with other genres, adding fresh perspectives to classic dream pop elements.

If you have a specific artist or album in mind, I can help you find more detailed information or context around them!

**Query 2:** Classic albums from the 90s — year filter + re-ranking

In [28]:
query_2 = 'What are the most influential rock albums of the 1990s?'

results_2 = hybrid_rag(
    query=query_2,
    collection=collection,
    keyword='rock',
    where_filter={'$and': [{'score': {'$gt': 7.5}}, {'year': {'$gte': 1990}}, {'year': {'$lt': 2000}}]},
    top_n_candidates=10,
    top_k_final=3
)

response_2 = generate_prompt(query_2, results_2)
answer_2 = client.responses.create(
    model=MODEL,
    instructions='You are a helpful assistant that provides information based on Pitchfork reviews.',
    input=[{'role': 'user', 'content': response_2}],
    max_output_tokens=500,
    temperature=1.0
).output_text
display(Markdown(answer_2))

One of the most influential rock albums of the 1990s is **"Spiderland"** by **Slint**, released in 1991 and rated a perfect **10.0** by Pitchfork. This album fundamentally reshaped the landscape of indie rock, leaving a lasting legacy that influenced various genres, including emo and post-rock. The record is noted for its unique approach to rock music, characterized by unpredictable song structures and a haunting atmosphere that has inspired countless artists in the years since its release.

Pitchfork's review emphasizes Slint's unusual musical journey, detailing how the band emerged from the post-hardcore scene in Louisville, Kentucky. Despite their limited initial exposure and live performances, "Spiderland" is now recognized as a classic that contributed to the foundations of the post-rock genre. The album's complex arrangements and lyrical depth created a template that artists like **Mogwai** and **Godspeed You! Black Emperor** would later build upon. The reviews also reflect on the band's posthumous recognition, noting that their influence grew significantly after their disbandment, with other major bands of the era—like Bush—seeking to capture similar sounds, which underscores Slint's impact on the rock scene of the 1990s and beyond.

Moreover, the reissue of "Spiderland" highlights its continuing relevance, as Pitchfork notes the inclusion of previously unreleased outtakes and insights into the band's little-acknowledged origins. This archival material not only sheds light on Slint's creative process but also further cements their status as pioneers in indie and alternative rock. The film "Breadcrumb Trail," featured in the reissue, serves as a poignant reminder of the band's artistic journey and their enduring legacy, making “Spiderland” a vital listen for anyone interested in the evolution of rock music in the 1990s.

**Query 3:** Electronic music — keyword only + re-ranking (no metadata filter)

In [29]:
query_3 = 'Recommend some groundbreaking electronic albums with experimental production.'

results_3 = hybrid_rag(
    query=query_3,
    collection=collection,
    keyword='electronic',
    where_filter=None,
    top_n_candidates=10,
    top_k_final=3
)

response_3 = generate_prompt(query_3, results_3)
answer_3 = client.responses.create(
    model=MODEL,
    instructions='You are a helpful assistant that provides information based on Pitchfork reviews.',
    input=[{'role': 'user', 'content': response_3}],
    max_output_tokens=500,
    temperature=1.0
).output_text
display(Markdown(answer_3))

For groundbreaking electronic albums that feature experimental production, consider the following three recommendations based on their unique contributions to the genre:

1. **"close to the noise floor - formative uk electronica 1975 - 1984"** by Various Artists (Score: 8.1):
   This compilation provides a fascinating overview of the early intersections of electronic music and home-studio experimentation in the UK. It highlights artists like Zorch and Ron Berry, whose works embody the avant-garde essence of their time, drifting into the realms of kosmische and proto-techno. The collection reflects how these pioneers laid the groundwork for future electronic movements, making it not only educational but also essential listening for aficionados of experimental soundscapes. The nostalgic echoes of early electronic experimentation resonate throughout, signaling a pivotal moment in music history.

2. **"flatland"** by Objekt (Score: 8.0):
   Objekt's debut LP is a striking exploration of contemporary electronic music that melds technical prowess with a raw, impulsive energy. As an artist who has a background in electronic engineering and has contributed to software development, Objekt's approach to production combines meticulous craftsmanship with a sense of innovation. The album captures the complexity of modern electronic sounds while maintaining an experimental edge. It challenges and expands upon traditional definitions of the genre, making it a vibrant and essential piece for anyone interested in the current landscape of electronic music.

3. **"4 parabolic mixes"** by Philip Jeck, Main, Oval, Henri Pousseur (Score: 7.3):
   Although this album scores slightly lower, it's noteworthy for its historical relevance. It revives the work of Henri Pousseur, a pioneer in electronic music, by showcasing reworkings of his early compositions. The experimental nature of these mixes provides valuable insight into the evolution of electronic sound, promoting an understanding of how far the genre has come since its inception. While it may resonate more as a nostalgic piece, the album captures the essence of electronic music's transformative journey.

These albums are significant not only for their scores but also for their contributions to the landscape of electronic music, showcasing how artists navigate and challenge the boundaries of sound. They embody the spirit of innovation and the experimental ethos that characterize groundbreaking electronic production.

## Summary

| Technique | What it does | Best for | Risk |
|-----------|-------------|----------|------|
| **Pure vector** | Semantic similarity across full corpus | Open-ended queries | May retrieve semantically close but unhelpful chunks |
| **Keyword filter** | Restricts to documents containing an exact term | Artist names, genre labels, specific albums | Misses synonyms; empty result if keyword is rare |
| **Metadata filter** | Restricts to chunks meeting numeric/string criteria | Quality gates (score), time ranges (year) | Tight filters reduce recall |
| **LLM re-ranking** | Model re-orders a candidate pool by relevance | Refining any of the above | Extra API call; negligible cost at small scale |
| **Hybrid (all three)** | Combines all layers | High-precision queries with clear constraints | Very tight keyword + metadata may return no candidates |

**Next steps:**
- Quantitative evaluation — measure how much each technique improves hit rate
- Try query intent extraction: use the LLM to parse `genre`, `score`, and `year` constraints from a natural-language query, then pass them as `where_filter` automatically